# 02 — Launch resumable monthly full-data exports

Run only after the pilot succeeds. Exporting six years with fine-scale terrain normalization can take substantial Earth Engine time. This notebook queues a few monthly tasks per run and records task IDs. Your account's task/compute quotas apply. It does not train a model or change the frozen study design.

In [ ]:
from pathlib import Path
import os, sys, subprocess, importlib.util, json
try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False
ROOT = next((p for p in [Path.cwd(), Path.cwd().parent, Path("/content/cht-soil-moisture-gapfilling")]
             if (p / "pyproject.toml").exists()), None)
if ROOT is None and IN_COLAB:
    ROOT = Path("/content/cht-soil-moisture-gapfilling")
    subprocess.run(["git", "clone", "--branch", "research-notebooks-v2", "--single-branch",
                    "https://github.com/Bipul-roy-AI/cht-soil-moisture-gapfilling.git", str(ROOT)], check=True)
if ROOT is None:
    raise RuntimeError("Open this notebook from the cloned repository, or run it in Colab.")
ROOT = ROOT.resolve()
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(ROOT) + "[gee,notebooks]"], check=True)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
print("Project:", ROOT)


In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = Path("/content/drive/MyDrive/CHT_SM_v2")
else:
    WORK = ROOT / "work_v2"
WORK.mkdir(parents=True, exist_ok=True)
print("Persistent work folder:", WORK)


In [ ]:
from chtsm.data import read_config, save_json
from chtsm.gee import month_windows, start_month_export
import ee
GEE_PROJECT_ID = "gee-projects-475506"
ee.Authenticate(); ee.Initialize(project=GEE_PROJECT_ID)
config = read_config(WORK / "study_config.json")
if config["relative_orbit"] is None:
    raise ValueError("Finish orbit selection in notebook 01 first.")
windows = month_windows(config)
print("Monthly tasks planned:", len(windows))
print(windows[:4], "...", windows[-2:])
ledger_path = WORK / "full_export_tasks.json"
ledger = json.loads(ledger_path.read_text()) if ledger_path.exists() else []
print("Tasks already recorded:", len(ledger))

In [ ]:
START_FULL_EXPORTS = False
NEXT_N_MONTHS = 3  # Launch small batches; monitor quota and completed outputs.
# Failed/cancelled tasks must be investigated. Remove ONLY their verified ledger
# entries before retrying so successful tasks are not accidentally duplicated.
existing = {r["start"] for r in ledger}
pending = [(a, b) for a, b in windows if a not in existing]
if START_FULL_EXPORTS:
    for start, end in pending[:NEXT_N_MONTHS]:
        prefix = "chtsm_v2_full_" + start[:7].replace("-", "")
        record = start_month_export(config, start, end, "CHT_SM_v2_exports", prefix)
        ledger.append(record)
        save_json(ledger_path, ledger)  # Persist after EACH successful submission.
        print("Submitted", prefix, record["task_id"])
else:
    print("Review pending tasks, then explicitly enable START_FULL_EXPORTS.")

In [ ]:
# Status only: no task is deleted or restarted automatically.
status = ee.data.getTaskStatus([r["task_id"] for r in ledger]) if ledger else []
for s in status:
    print(s.get("id"), s.get("state"), s.get("error_message", ""))

Move only completed `chtsm_v2_full_*.csv` files into `MyDrive/CHT_SM_v2/exports/`. Drive may split a large export into multiple CSV parts; retain every part with that prefix. Do not mix pilot exports, duplicate downloads, old NDVI-renamed tables, different orbit settings or product versions. Notebook 03 rejects duplicate cells/dates, mixed provenance and incomplete calendars. The complete export is required for the declared six-year study; reduce the study prospectively if resources require a smaller feasibility design.